# MICrONS: root IDs at the nucleus + cell volumes (seg_m1300)

Run the cells top to bottom (Runtime → Run all). Everything is anonymous: no CAVE token, no Google Cloud login.
Google Drive is mounted only to keep checkpoints and the results, so a dropped runtime resumes where it stopped.

**Part A.** For the ~14,300 nuclei with root ID 0 in `nucleus_detection_v0.csv`, read the `seg_m1300` segmentation in a small cube around the nucleus centre and take the most common non-zero label. A random sample of nuclei that *do* have a root is read the same way, to check that the table and the segmentation agree.

**Part B.** For every root ID (table + Part A), download its `seg_m1300` mesh and compute its volume (divergence theorem, the same formula µJump's *Compute volume* button uses). Default: the coarsest level of detail, which is fast; a check on 40 cells reports how far it is from the next finer level.

Outputs (in `MyDrive/microns_m1300_roots/`):
- `nuc_roots_m1300.csv` — one row per nucleus: final root, where it came from, vote share.
- `cell_volumes_m1300.csv` — one row per root: volume (µm³), LOD, vertices, how many nuclei share it.
- `nucleus_cell_volumes_m1300.csv` and an Excel-friendly `…_excel.csv` (semicolon, decimal comma).

Send the first three files (or put them in the repo folder `data/nuc_roots/m1300/`) and they will be packed into µJump and the VR viewer.


In [ ]:
!pip -q install cloud-volume


In [ ]:
# ── settings ─────────────────────────────────────────────────────────────
SEG = "precomputed://gs://iarpa_microns/minnie/minnie65/seg_m1300"
OUT_DIR = "/content/drive/MyDrive/microns_m1300_roots"   # results + checkpoints
USE_DRIVE = True          # False: keep everything in /content (lost if the runtime dies)
N_THREADS = 10           # 10 = no "Connection pool is full" warnings
CUBE_HALF_UM = (1.5, 1.5, 0.8)   # half-size of the cube read around each nucleus centre (x, y, z), µm
MAX_XY_RES_NM = 64        # read the segmentation at the coarsest level whose x/y voxel is <= this
QC_SAMPLE = 1000          # nuclei WITH a table root, re-read from the segmentation as a check
VOL_LOD = -1              # -1 = coarsest level of detail (fast); 0 = finest (slow, large meshes)
LOD_CHECK_N = 40
MIN_VERTS = 2000          # meshes with fewer vertices at the chosen level are re-read finer
VOL_TAG = "vol3"          # checkpoint name for volumes (the first run's "vol" files had rounded IDs)          # cells whose volume is also computed one level finer, to report the difference
BATCH = 2000              # rows per checkpoint file
RUN_PART_A = True
READ_ALL = True           # also read the segmentation at every nucleus that HAS a table root: 16 % of
                          # table roots are not the seg_m1300 root (and have no mesh there)
MIN_SHARE = 0.5           # the label at the nucleus must hold at least this share of the labelled voxels
RUN_PART_B = True

import os, sys, time, math, json, glob, threading, random
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# wider HTTP connection pool, before any CloudVolume exists (otherwise >10 threads re-handshake)
import requests.adapters as _ad
_POOL = max(N_THREADS * 2, 20)
_ad.DEFAULT_POOLSIZE = _POOL
_orig_init = getattr(_ad.HTTPAdapter, "_orig_init", _ad.HTTPAdapter.__init__)
def _init(self, *args, **kw):
    if not args:
        kw["pool_connections"] = max(kw.get("pool_connections") or _POOL, _POOL)
        kw["pool_maxsize"] = max(kw.get("pool_maxsize") or _POOL, _POOL)
    return _orig_init(self, *args, **kw)
_ad.HTTPAdapter._orig_init = _orig_init
_ad.HTTPAdapter.__init__ = _init

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted (" + str(e) + ") - saving to /content instead")
        OUT_DIR = "/content/microns_m1300_roots"
else:
    OUT_DIR = "/content/microns_m1300_roots"
CKPT = os.path.join(OUT_DIR, "checkpoints")
os.makedirs(CKPT, exist_ok=True)
print("results go to", OUT_DIR)


In [ ]:
# ── the nucleus table ────────────────────────────────────────────────────
# Looks for nucleus_detection_v0.csv in OUT_DIR first, then asks you to upload it.
cands = [os.path.join(OUT_DIR, "nucleus_detection_v0.csv"), "/content/nucleus_detection_v0.csv"]
path = next((p for p in cands if os.path.exists(p)), None)
if path is None:
    from google.colab import files
    print("Upload nucleus_detection_v0.csv (root_id,id,volume,pt_position_x,pt_position_y,pt_position_z, positions in nm)")
    up = files.upload()
    name = next(iter(up))
    path = os.path.join(OUT_DIR, "nucleus_detection_v0.csv")
    with open(path, "wb") as f: f.write(up[name])
# Excel can mangle a row into thousands-separated digits (e.g. "8,646,911,367,..."); such rows are
# skipped and counted, not guessed at.
bad = []
nuc = pd.read_csv(path, encoding="utf-8-sig", dtype=str, engine="python", on_bad_lines=lambda l: bad.append(l) or None)
n0 = len(nuc)
ok = nuc["root_id"].str.fullmatch(r"\s*\d+\s*", na=False) & nuc["id"].str.fullmatch(r"\s*\d+\s*", na=False)
nuc = nuc[ok].copy()
nuc["root_id"] = np.array([int(v) for v in nuc["root_id"]], dtype=np.uint64)   # exact: never via float
for c in ["id", "volume", "pt_position_x", "pt_position_y", "pt_position_z"]:
    nuc[c] = pd.to_numeric(nuc[c], errors="coerce")
nuc = nuc.dropna(subset=["id", "pt_position_x", "pt_position_y", "pt_position_z"])
nuc["id"] = nuc["id"].astype("int64")
print(len(bad), "malformed rows skipped;", n0 - len(nuc), "rows with missing numbers dropped")
nuc = nuc.rename(columns={"id": "nucleus_id", "volume": "nucleus_volume_um3"})
nuc = nuc.drop_duplicates("nucleus_id").reset_index(drop=True)
print(len(nuc), "nuclei;", int((nuc.root_id == 0).sum()), "with root ID 0")


In [ ]:
# ── the segmentation ─────────────────────────────────────────────────────
from cloudvolume import CloudVolume
_tl = threading.local()
def seg_cv():
    if not hasattr(_tl, "cv"):
        _tl.cv = CloudVolume(SEG, mip=MIP, use_https=True, fill_missing=True, progress=False,
                             parallel=1, bounded=False, cache=False)
    return _tl.cv
def mesh_cv():
    if not hasattr(_tl, "mcv"):
        _tl.mcv = CloudVolume(SEG, use_https=True, progress=False, parallel=1, cache=False)
    return _tl.mcv

cv0 = CloudVolume(SEG, use_https=True, progress=False)
mips = list(cv0.meta.available_mips)
for m in mips:
    print("mip", m, "resolution nm", list(cv0.meta.resolution(m)), "chunk", list(cv0.meta.chunk_size(m)))
MIP = max([m for m in mips if cv0.meta.resolution(m)[0] <= MAX_XY_RES_NM] or [0])
RES = np.array(cv0.meta.resolution(MIP), dtype=float)
HALF = np.maximum(1, np.round(np.array(CUBE_HALF_UM) * 1000 / RES)).astype(int)
print("reading labels at mip", MIP, RES, "nm; cube half-size", HALF, "voxels")

def root_at(x_nm, y_nm, z_nm):
    # most common non-zero label in a cube around the point -> (root, share of non-zero voxels, share non-zero)
    c = np.round(np.array([x_nm, y_nm, z_nm]) / RES).astype(int)
    lo, hi = c - HALF, c + HALF + 1
    a = np.asarray(seg_cv()[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]).reshape(-1)
    nz = a[a != 0]
    if nz.size == 0:
        return 0, 0.0, 0.0
    v, n = np.unique(nz, return_counts=True)
    k = int(np.argmax(n))
    return int(v[k]), float(n[k] / nz.size), float(nz.size / a.size)

# Self-test: verified by hand on 2026-09-10 (µJump's segread note) - nucleus 373879 lies in root 864691136090135607.
row = nuc[nuc.nucleus_id == 373879].iloc[0]
r, share, filled = root_at(row.pt_position_x, row.pt_position_y, row.pt_position_z)
print("self-test:", "OK" if r == 864691136090135607 else "FAIL", r, round(share, 3))
assert r == 864691136090135607, "the coordinate handling is wrong - do not use the output"


In [ ]:
# ── Part A: root IDs at the nucleus ──────────────────────────────────────
def run_batches(tag, df, fn, cols):
    # fn(row) -> tuple matching cols; checkpoint every BATCH rows; resumes from existing files
    out = []
    nb = math.ceil(len(df) / BATCH) if len(df) else 0
    t0 = time.time()
    for b in range(nb):
        f = os.path.join(CKPT, f"{tag}_{b:04d}.csv")
        if os.path.exists(f):
            d = pd.read_csv(f, dtype={c: "uint64" for c in cols if c.endswith("root_id")})
            d["error"] = d["error"].fillna("").astype(str); out.append(d); continue
        part = df.iloc[b * BATCH:(b + 1) * BATCH]
        res = [None] * len(part)
        with ThreadPoolExecutor(N_THREADS) as ex:
            futs = {ex.submit(fn, r): i for i, r in enumerate(part.itertuples(index=False))}
            for fu in as_completed(futs):
                i = futs[fu]
                try: res[i] = fu.result()
                except Exception as e: res[i] = fn.fail(part.iloc[i], e)
        d = pd.DataFrame(res, columns=cols)
        for c in cols:   # uint64 in every frame: concat of uint64 + int64 silently gives float64 and rounds the IDs
            if c.endswith("root_id"): d[c] = np.array([int(x) for x in d[c]], dtype=np.uint64)
        d.to_csv(f, index=False); out.append(d)
        el = time.time() - t0
        print(f"{tag}: batch {b + 1}/{nb} done, {el / 60:.1f} min so far")
    res = pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=cols)
    for c in cols:
        if c.endswith("root_id") and len(res): assert res[c].dtype == np.uint64, c + " lost its integer type"
    return res

A_COLS = ["nucleus_id", "voxel_root_id", "vote_share", "filled_share", "error"]
def a_fn(r):
    v, s, f = root_at(r.pt_position_x, r.pt_position_y, r.pt_position_z)
    return (int(r.nucleus_id), v, round(s, 4), round(f, 4), "")
a_fn.fail = lambda r, e: (int(r.nucleus_id), 0, 0.0, 0.0, str(e)[:200])

if RUN_PART_A:
    zero = nuc[nuc.root_id == 0]
    rng = np.random.default_rng(0)
    qc = nuc[nuc.root_id != 0].sample(min(QC_SAMPLE, int((nuc.root_id != 0).sum())), random_state=0)
    qc_res = run_batches("qc", qc, a_fn, A_COLS).merge(qc[["nucleus_id", "root_id"]], on="nucleus_id")
    agree = (qc_res.voxel_root_id == qc_res.root_id).mean()
    print(f"QC: the segmentation agrees with the table root for {agree * 100:.1f} % of {len(qc_res)} nuclei")
    zres = run_batches("zero", zero, a_fn, A_COLS)
    print(f"Part A: {int((zres.voxel_root_id != 0).sum())} of {len(zres)} root-0 nuclei have a cell at the nucleus;",
          int((zres.error != '').sum()), "read errors")
    qc_res.to_csv(os.path.join(OUT_DIR, "qc_table_vs_voxels.csv"), index=False)
    if READ_ALL:
        tres = run_batches("tab", nuc[nuc.root_id != 0], a_fn, A_COLS)
        tm = tres.merge(nuc[["nucleus_id", "root_id"]], on="nucleus_id")
        print(f"All table roots: the segmentation at the nucleus holds a DIFFERENT root for {(tm.voxel_root_id != tm.root_id).mean() * 100:.1f} % of {len(tm)}")


In [ ]:
# ── the per-nucleus root table ───────────────────────────────────────────
roots = nuc[["nucleus_id", "root_id", "nucleus_volume_um3", "pt_position_x", "pt_position_y", "pt_position_z"]].copy()
# Root IDs are 18-digit integers; a float64 holds only ~16 digits, so they are kept in a
# uint64 numpy array and NEVER assigned through .loc (which can turn the column into floats).
rid = nuc.root_id.to_numpy(dtype=np.uint64).copy()
src = np.where(rid != 0, "table", "none").astype(object)
vs = np.full(len(rid), np.nan)
if RUN_PART_A:
    zmap = {int(a): (int(b), float(c)) for a, b, c in zip(zres.nucleus_id, zres.voxel_root_id, zres.vote_share)}
    tmap = {int(a): (int(b), float(c)) for a, b, c in zip(tres.nucleus_id, tres.voxel_root_id, tres.vote_share)} if READ_ALL else {}
    for i, n_ in enumerate(roots.nucleus_id.to_numpy()):
        if rid[i] == 0 and int(n_) in zmap:
            r_, s_ = zmap[int(n_)]
            rid[i] = np.uint64(r_); vs[i] = s_
            if r_ != 0: src[i] = "voxels at nucleus"
        elif rid[i] != 0 and int(n_) in tmap:
            r_, s_ = tmap[int(n_)]
            vs[i] = s_
            if r_ != 0 and r_ != int(rid[i]) and s_ >= MIN_SHARE:
                rid[i] = np.uint64(r_); src[i] = "voxels (table root outdated)"
roots["root_id"] = rid; roots["root_source"] = src; roots["vote_share"] = vs
assert roots.root_id.dtype == np.uint64
t = roots.root_source.to_numpy() == "table"
assert (roots.root_id.to_numpy()[t] == nuc.root_id.to_numpy(dtype=np.uint64)[t]).all(), "root IDs changed - precision lost"
share = roots[roots.root_id != 0].groupby("root_id").size()
roots["nuclei_on_root"] = roots.root_id.map(share).fillna(0).astype(int)
roots.to_csv(os.path.join(OUT_DIR, "nuc_roots_m1300.csv"), index=False)
print(roots.root_source.value_counts().to_string())
print(int((roots.nuclei_on_root > 1).sum()), "nuclei share their root with another nucleus (merges)")


In [ ]:
# ── Part B: cell volumes from the meshes ─────────────────────────────────
def mesh_volume(m):
    v = np.asarray(m.vertices, dtype=np.float64); f = np.asarray(m.faces, dtype=np.int64)
    if len(f) == 0: return 0.0, v
    a, b, c = v[f[:, 0]], v[f[:, 1]], v[f[:, 2]]
    return abs(np.einsum("ij,ij->i", a, np.cross(b, c)).sum()) / 6.0 / 1e9, v   # nm^3 -> µm^3

def get_mesh(sid, lod):
    mm = mesh_cv().mesh.get(int(sid), lod=lod)
    if isinstance(mm, dict): mm = mm.get(int(sid)) or next(iter(mm.values()))
    return mm

pos = roots[roots.root_id != 0].groupby("root_id")[["pt_position_x", "pt_position_y", "pt_position_z"]].first()
B_COLS = ["root_id", "cell_volume_um3", "lod", "n_lods", "n_vertices", "nucleus_in_bbox", "error"]
def b_fn(r):
    sid = int(r.root_id)
    man = mesh_cv().mesh.get_manifest(sid)
    if man is None: return (sid, np.nan, -1, 0, 0, False, "no mesh in seg_m1300")
    lod = man.num_lods - 1 if VOL_LOD == -1 else min(VOL_LOD, man.num_lods - 1)
    vol, v = mesh_volume(get_mesh(sid, lod))
    # a small segment's coarsest level can be a handful of vertices; go finer until it has MIN_VERTS
    while len(v) < MIN_VERTS and lod > 0:
        lod -= 1; vol, v = mesh_volume(get_mesh(sid, lod))
    p = np.array([r.pt_position_x, r.pt_position_y, r.pt_position_z])
    inb = bool(len(v) and np.all(p >= v.min(0) - 2000) and np.all(p <= v.max(0) + 2000))
    return (sid, round(vol, 3), lod, man.num_lods, len(v), inb, "")
b_fn.fail = lambda r, e: (int(r.root_id), np.nan, -1, 0, 0, False, str(e)[:200])

if RUN_PART_B:
    todo_all = pos.reset_index()
    assert todo_all.root_id.dtype == np.uint64 and set(todo_all.root_id.tolist()) <= set(roots.root_id.tolist())
    # volumes already computed in an earlier run (checkpoints "vol2") are reused, not re-read
    old = [pd.read_csv(f, dtype={"root_id": "uint64"}) for f in sorted(glob.glob(os.path.join(CKPT, "vol2_*.csv")))]
    cache = pd.concat(old, ignore_index=True) if old else pd.DataFrame(columns=B_COLS)
    if len(cache): cache["error"] = cache["error"].fillna("").astype(str); cache["root_id"] = np.array([int(x) for x in cache.root_id], dtype=np.uint64)
    have = set(cache.root_id.tolist())
    todo = todo_all[~todo_all.root_id.isin(have)].reset_index(drop=True)
    print(len(todo_all), "roots;", len(todo_all) - len(todo), "volumes reused from the earlier run;", len(todo), "to read")
    # units check before the long run: the nucleus must lie inside its own cell's mesh box
    test = [b_fn(r) for r in todo_all.head(10).itertuples(index=False)]
    ok = [t for t in test if not t[6]]
    print("units check:", sum(t[5] for t in ok), "of", len(ok), "meshes contain their nucleus;", [round(t[1], 1) for t in ok][:5], "µm³")
    assert ok and sum(t[5] for t in ok) >= 0.8 * len(ok), "mesh coordinates are not in nm or not aligned - stop"
    # how much the coarse level changes the volume
    diffs = []
    for r in todo_all.sample(min(LOD_CHECK_N, len(todo_all)), random_state=1).itertuples(index=False):
        try:
            man = mesh_cv().mesh.get_manifest(int(r.root_id))
            if man is None or man.num_lods < 2: continue
            lc = man.num_lods - 1 if VOL_LOD == -1 else min(VOL_LOD, man.num_lods - 1)
            if lc == 0: continue
            v1, _ = mesh_volume(get_mesh(r.root_id, lc)); v0, _ = mesh_volume(get_mesh(r.root_id, lc - 1))
            if v0 > 0: diffs.append(100 * (v1 - v0) / v0)
        except Exception as e:
            pass
    if diffs:
        print(f"LOD check on {len(diffs)} cells: chosen level vs one finer, median {np.median(diffs):+.1f} %, 90th pct |diff| {np.percentile(np.abs(diffs), 90):.1f} %")
    new = run_batches(VOL_TAG, todo, b_fn, B_COLS) if len(todo) else pd.DataFrame(columns=B_COLS)
    vols = pd.concat([cache[cache.root_id.isin(set(todo_all.root_id.tolist()))][B_COLS], new[B_COLS]], ignore_index=True)
    vols["root_id"] = np.array([int(x) for x in vols.root_id], dtype=np.uint64)
    vols["error"] = vols["error"].fillna("").astype(str)
    assert set(vols.root_id.tolist()) == set(todo_all.root_id.tolist()), "volume root IDs do not match - precision lost"
    vols["root_id"] = vols.root_id.astype("uint64")
    vols["nuclei_on_root"] = vols.root_id.map(share).fillna(0).astype(int)
    vols.to_csv(os.path.join(OUT_DIR, "cell_volumes_m1300.csv"), index=False)
    print(int(vols.cell_volume_um3.notna().sum()), "of", len(vols), "roots have a volume;",
          int((vols.error != "").sum()), "errors;", int((~vols.nucleus_in_bbox & (vols.error == "")).sum()), "meshes not around their nucleus")


In [ ]:
# ── per nucleus, with flags; plus an Excel-friendly copy ─────────────────
out = roots.copy()
if RUN_PART_B:
    out = out.merge(vols[["root_id", "cell_volume_um3", "lod", "n_vertices", "error"]], on="root_id", how="left")
    flags = []
    for r in out.itertuples(index=False):
        f = []
        if r.root_id == 0: f.append("no cell at nucleus")
        if r.nuclei_on_root > 1: f.append(f"merge ({r.nuclei_on_root} nuclei)")
        cv_ = r.cell_volume_um3
        if isinstance(cv_, float) and not math.isnan(cv_) and cv_ < (r.nucleus_volume_um3 or 0): f.append("cell smaller than nucleus (fragment?)")
        if isinstance(r.error, str) and r.error: f.append(r.error)
        flags.append("; ".join(f))
    out["flags"] = flags
out.to_csv(os.path.join(OUT_DIR, "nucleus_cell_volumes_m1300.csv"), index=False)
x = out.copy(); x["root_id"] = x.root_id.astype(str)
x.to_csv(os.path.join(OUT_DIR, "nucleus_cell_volumes_m1300_excel.csv"), index=False, sep=";", decimal=",")
print("saved to", OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))
